# Main file 

05.11.2025

Written by: Arnav Kumar Upadhyay, Cedric Meyer adapted from MATLAB code of Jan Spindler DOI-https://doi.org/10.1021/acssynbio.4c00578

This script is the main script for the TXTL v2 model.


In [ ]:
# Packages functions and data types --------
include("packages.jl") 

# Sequences 
sequence_file	    = "Ecoli_K12_MG1655_geneset_with_welch_2009.fa"    # "lacS_background.fa", "Ecoli_K12_MG1655_geneset_cds_hpaBC_ddc.fa", "Ecoli_example_sequences.fa"
hybrid_file		    = "lacS.txt"
seqs_to_use 	    = (1,2323)                                 # Range of sequences in Fasta file. Takes all sequences from sequence a to sequence b
num_background_genes = 75
heterologous_genes  = ["hpaBC"]#,"ddc"]

# initialization of data containers
scale = txtl_types.init_scale()         # struct that makes it possible to influence rates by scaling them

µ = 1                    # A growth rate (0.1 to 1.8) can be chosen and the proteins of the TXTL machinery will be tuned according to that growth rate and the processivity of the RNA polymerase

## flags for efficiency factors
positive_aminoacid	= true 
anti_shinedalgarno	= false      # If no hybridization file is available put flag to false. We apply the intaRNA program's intarna function (q: aSD, t: sequence, SeedBP: 5, out: tMinE) to generate a hybridization profile along the sequence. Link to the website: https://rna.informatik.uni-freiburg.de/IntaRNA/Input.jsp;jsessionid=2AE257E91E7276283F22CD77A5A8AA03
dynamic_efficiency	= true 
save_data           = false

flags = txtl_types.flags(positive_aminoacid, anti_shinedalgarno, dynamic_efficiency)

# Data import --------
# mRNA concentrations table (used as µM throughout the model)
mRNA 		= to_DataFrame(readmatrix("../data/input/mRNA_concentrations.xlsx", "mRNA_concentration", "A1:B2324"))
mRNA.mRNA 	= Float64.(mRNA.mRNA);

# Global controller: scales ALL mRNA concentrations used in the simulation
# Example: 0.5 halves total mRNA, 2.0 doubles total mRNA
mRNA_global_multiplier = 0.5
mRNA.mRNA .*= mRNA_global_multiplier

background_genes = String.(first(mRNA.gene, num_background_genes))
genes = vcat(heterologous_genes, background_genes)
println("Number of sequences: $(length(genes))")

# Keep only requested genes
mRNA = subset(mRNA, :gene => ByRow(in(genes)))

# Total mRNA concentration for the simulated genes (µM)
println("🔧 Total mRNA concentration (after scaling): $(sum(mRNA.mRNA)) µM")

## Import information about model sp ecies and sequences 
seqs, seq_names, hyb_file_dict, species_and_seqs, idx = species_and_sequences(genes, µ, sequence_file, hybrid_file, flags, seqs_to_use)

## Calculate efficiency factors 
efac_pos_aa_dict, efac_int_sd_dict, dynamic_efficiencies = efficiency_factor_initialization(seqs, seq_names, hyb_file_dict, species_and_seqs, flags)

## calculation of initial concentrations
y0, initial_concentrations = calculation_of_initial_concentration(species_and_seqs, seqs, seq_names, scale, idx);

## calculate and load kinetic parameters
p_met, p_seq, p_dyn = parameters(initial_concentrations, species_and_seqs, scale, µ);

## Release factor balance
RF_balance!(y0, initial_concentrations)

# Preintegration
## Definition of mass matrix
conserved = [idx.EFG, idx.EFD] # EFG, EFTu_GDP - Conserved species, Algebraic equations are given by mass balances

M = Matrix{Float64}(I, length(y0), length(y0)) # Creates unity matrix in the size of y0

M[conserved, :] .= 0; # setting mass matrix to zero to  remove time derivatives from those variables. This converts these differential equations into algebraic constraints


tags_pre = txtl_types.data_struct_preintegration(scale, idx, species_and_seqs, flags, dynamic_efficiencies, initial_concentrations, p_met, p_seq, p_dyn, mRNA.mRNA[1])

## Integration of the DAE system without DNA in the system 
y_steady_state = preintegration(tags_pre, y0, M);

# ===== DIAGNOSTIC: Check preintegration results =====
println("\n" * "="^80)
println("🔍 PREINTEGRATION DIAGNOSTICS")
println("="^80)
println("Vector length: $(length(y_steady_state))")
println("Minimum value: $(minimum(y_steady_state))")
println("Maximum value: $(maximum(y_steady_state))")
println("Non-finite entries: $(count(!isfinite, y_steady_state))")
println("Values < -1e-9: $(count(x -> x < -1e-9, y_steady_state))")
println("Values < 0: $(count(x -> x < 0, y_steady_state))")

# Find which species are negative
negative_indices = findall(x -> x < -1e-9, y_steady_state)
if !isempty(negative_indices)
    println("\n⚠️  NEGATIVE SPECIES DETECTED:")
    for idx in negative_indices
        if idx <= length(tags_pre.species_and_sequences.Tag)
            tag = tags_pre.species_and_sequences.Tag[idx]
            value = y_steady_state[idx]
            println("  idx=$(idx) tag=$(tag) value=$(value)")
        else
            println("  idx=$(idx) (codon position) value=$(y_steady_state[idx])")
        end
    end
else
    println("\n✅ No negative species detected (threshold: -1e-9)")
end
println("="^80 * "\n")

# Elongation factor balance 
EF_balance!(tags_pre, y0, y_steady_state)


# Build states vector for multiple sequences 
y0_all = combined_states_vector(tags_pre, y_steady_state, seqs, seq_names)

# ===== DIAGNOSTIC: Check combined states vector =====
println("\n" * "="^80)
println("🔍 COMBINED STATES VECTOR DIAGNOSTICS")
println("="^80)
println("Vector length: $(length(y0_all))")
println("Minimum value: $(minimum(y0_all))")
println("Maximum value: $(maximum(y0_all))")
println("Non-finite entries: $(count(!isfinite, y0_all))")
println("Values < -1e-9: $(count(x -> x < -1e-9, y0_all))")
println("Values < 0: $(count(x -> x < 0, y0_all))")

# Find which species are negative
negative_indices_all = findall(x -> x < -1e-9, y0_all)
if !isempty(negative_indices_all)
    println("\n⚠️  NEGATIVE SPECIES IN y0_all:")
    # Structure: [shared metabolites] [seq1: S70, rna, PRT, PTC, codons] [seq2: ...] ...
    n_metab_shared = length(tags_pre.species_and_sequences.Tag) - 4  # Shared metabolite pool (excluding last 4)
    
    # Calculate per-sequence structure
    seq_lengths = [length(seqs[seq_name].Codon) - 1 for seq_name in seq_names]
    n_per_seq = [4 + seq_lengths[i] for i in 1:length(seq_names)]  # S70, rna, PRT, PTC + codons per sequence
    
    for idx in negative_indices_all[1:min(20, length(negative_indices_all))]  # Show first 20
        if idx <= n_metab_shared
            tag = tags_pre.species_and_sequences.Tag[idx]
            println("  idx=$(idx) tag=$(tag) value=$(y0_all[idx]) [shared metabolite pool]")
        else
            # Find which sequence this belongs to
            pos = idx - n_metab_shared
            seq_num = 1
            cumsum = 0
            for i in 1:length(n_per_seq)
                if pos <= cumsum + n_per_seq[i]
                    seq_num = i
                    pos_in_seq = pos - cumsum
                    break
                end
                cumsum += n_per_seq[i]
            end
            
            if pos_in_seq <= 4
                tags_seq = ["S70", "rna", "PRT", "PTC"]
                println("  idx=$(idx) seq=$(seq_num) ($(seq_names[seq_num])) $(tags_seq[pos_in_seq]) value=$(y0_all[idx])")
            else
                codon_pos = pos_in_seq - 4
                println("  idx=$(idx) seq=$(seq_num) ($(seq_names[seq_num])) codon_pos=$(codon_pos) value=$(y0_all[idx])")
            end
        end
    end
    if length(negative_indices_all) > 20
        println("  ... and $(length(negative_indices_all) - 20) more negative values")
    end
else
    println("\n✅ No negative species detected in y0_all (threshold: -1e-9)")
end
println("="^80 * "\n")


# Prepare per-sequence tags (for use in ODE)
local_tags_list = generate_local_tags(seqs, seq_names, tags_pre, efac_pos_aa_dict, efac_int_sd_dict, mRNA);


# Integration
# PERFORMANCE FIX: Use Diagonal mass matrix instead of dense matrix
# For 100 sequences with ~22k states, dense matrix would be ~4GB and cause O(n³) scaling
n_all = length(y0_all)
mass_diag_all = ones(Float64, n_all)
mass_diag_all[conserved] .= 0.0
M = Diagonal(mass_diag_all)  # Much more memory-efficient: O(n) instead of O(n²)

tspan 	= (0.0, 100.0) ## Define time span for the simulation

p = generate_p(local_tags_list)

f 		= ODEFunction(ODE_parallel_distributed!, mass_matrix = M) ## Define ODEFunction to make use of mass matrix

prob 	= ODEProblem(f, y0_all, tspan, p); ## Create an ODE problem



# Tripwire callback: terminate immediately if any state becomes NaN/Inf or very negative.
# When it triggers, we also decode the global index into (shared tag) or (gene + codon position).

function describe_state_index(i::Int, tags_pre, seqs, seq_names)
    # combined_states_vector layout:
    #   u = [shared_metabolites;
    #        for each sequence j: (4 output vars) + (ribosome positions...)]
    n_metab_shared = length(tags_pre.species_and_sequences.Tag) - 4

    # Shared pool species
    if i <= n_metab_shared
        tag = tags_pre.species_and_sequences.Tag[i]
        return "shared pool: tag=$(tag)"
    end

    # Sequence-specific block
    pos = i - n_metab_shared
    out_tags = tags_pre.species_and_sequences.Tag[n_metab_shared+1 : n_metab_shared+4]
    seq_ribo_lens = [length(seqs[name].Codon) - 1 for name in seq_names]

    cumsum = 0
    for (j, Lr) in enumerate(seq_ribo_lens)
        block = 4 + Lr
        if pos <= cumsum + block
            pos_in_seq = pos - cumsum
            gene = seq_names[j]

            if pos_in_seq <= 4
                return "sequence=$(j) gene=$(gene) output_var=$(out_tags[pos_in_seq])"
            else
                ribo_pos = pos_in_seq - 4
                codon = (1 <= ribo_pos <= Lr) ? seqs[gene].Codon[ribo_pos] : "(out of range)"
                return "sequence=$(j) gene=$(gene) ribosome_pos=$(ribo_pos) codon=$(codon)"
            end
        end
        cumsum += block
    end

    return "(could not map index i=$(i))"
end

# Tripwire policy:
# - If NaN/Inf: terminate immediately.
# - If negative but not catastrophic: print location, clamp to 0, continue.
# - If very negative: print location and terminate.

const neg_tol_warn  = -1e-6   # below this we consider it "negative"
const neg_tol_hard  = -1e-2   # terminate if value < neg_tol_hard

function condition_tripwire(u, t, integrator)
    for i in eachindex(u)
        if !isfinite(u[i]) || u[i] < neg_tol_warn
            return true
        end
    end
    return false
end

function affect_tripwire!(integrator)
    u = integrator.u
    t = integrator.t

    for i in eachindex(u)
        if !isfinite(u[i])
            println("\n⚠️  TRIPWIRE TRIGGERED at t=$(t):")
            println("  Non-finite value detected at index $(i)")
            println("  Where: " * describe_state_index(i, tags_pre, seqs, seq_names))
            println("  Value: $(u[i])")
            terminate!(integrator)
            return
        elseif u[i] < neg_tol_warn
            # negatives happen with stiff solvers; clamp unless it's catastrophically negative
            if u[i] >= neg_tol_hard
                println("\n⚠️  TRIPWIRE: negative clamped at t=$(t):")
                println("  index $(i)")
                println("  Where: " * describe_state_index(i, tags_pre, seqs, seq_names))
                println("  Value: $(u[i]) (clamped to 0; hard_tol=$(neg_tol_hard))")
                u[i] = 0.0
                return
            else
                println("\n⚠️  TRIPWIRE TRIGGERED at t=$(t):")
                println("  Catastrophic negative value detected at index $(i)")
                println("  Where: " * describe_state_index(i, tags_pre, seqs, seq_names))
                println("  Value: $(u[i]) (warn_tol=$(neg_tol_warn), hard_tol=$(neg_tol_hard))")
                terminate!(integrator)
                return
            end
        end
    end
end

cb = DiscreteCallback(condition_tripwire, affect_tripwire!)

# PERFORMANCE FIX: Optimize save settings for large state vectors
# With ~22k states, saving every step is extremely expensive
# Only save start and end unless dense output is needed
sol = solve(prob, FBDF(autodiff=false),
            reltol = 1e-8, abstol = 1e-6,
            callback = cb,
            save_everystep = false, save_start = false, save_end = true,
            dense = false) ## Solve ODE system (final state only)

# Check solver return code
println("\n" * "="^80)
println("🔍 SOLVER STATUS")
println("="^80)
println("Return code: $(sol.retcode)")
if sol.retcode != :Success
    println("⚠️  WARNING: Solver did not complete successfully!")
    println("  This may indicate numerical issues or constraint violations.")
else
    println("✅ Solver completed successfully")
end
println("="^80 * "\n")

println("Integration completed!")

🔧 Julia started with 8 threads
Number of sequences: 2

⏳ Integrating ODE system (threaded, 2 sequences)...
Integration completed!
